# System Dynamics, Modes, and Observability / Controllability

*Supplement to the **Kalman Filter Boot Camp**. Bridges [01 · Continuous State-Space Models](01_Continuous_State_Space_Models.ipynb) and [03 · Discrete-Time State-Space Models](03_Discrete_Time_State_Space_Models_and_Simulation.ipynb) into the systems-theory results the Kalman filter quietly depends on.*

**Style:** every equation is followed by a plain-language paraphrase (→). Extra intuition beyond the lectures is flagged **→ Intuition**.

### 🧩 Time-Domain Response of a Continuous LTI System

- Start from the linear model $\dot{x}(t) = A\,x(t) + B\,u(t)$ with initial state $x(t_0)$.

- The exact solution is

$$
x(t) = e^{A(t-t_0)}\,x(t_0) + \int_{t_0}^{t} e^{A(t-\tau)} B\, u(\tau)\, d\tau .
$$

  → The state now is the **initial state carried forward** by $e^{A(t-t_0)}$, **plus** the accumulated effect of every past input, each input pushed forward by the same exponential from the instant it acted.

- The matrix exponential is defined by the same power series as the scalar one:

$$
e^{A t} = I + A t + \tfrac{1}{2!}A^2 t^2 + \tfrac{1}{3!}A^3 t^3 + \cdots = \sum_{k=0}^{\infty}\frac{(A t)^k}{k!}.
$$

  → It is **not** $e$ raised element-by-element. It is the operator built so that $\frac{d}{dt}e^{At}=A\,e^{At}$ — i.e. so that it exactly solves the dynamics.

### 🧩 The State-Transition Matrix $\Phi$

- Define $\Phi(t,t_0) = e^{A(t-t_0)}$, the **state-transition matrix (STM)**.

- It obeys:

$$
\Phi(t_0,t_0) = I, \qquad \Phi(t_2,t_0) = \Phi(t_2,t_1)\,\Phi(t_1,t_0), \qquad \Phi^{-1}(t,t_0) = \Phi(t_0,t).
$$

  → Transitioning over zero time does nothing ($I$); transitioning in two hops equals one big hop; every transition is reversible (run time backwards).

- **→ Intuition:** $\Phi$ is the "time machine" for the state vector. The Kalman filter's whole prediction step is nothing but applying a discrete $\Phi$ to the estimate and its covariance.

### 🧩 Modes and the Eigendecomposition

- If $A$ is diagonalizable, $A = V \Lambda V^{-1}$ with $\Lambda = \mathrm{diag}(\lambda_1,\dots,\lambda_n)$ (eigenvalues) and columns of $V$ the eigenvectors. Then

$$
e^{A t} = V\, e^{\Lambda t}\, V^{-1} = V \,\mathrm{diag}\!\big(e^{\lambda_1 t},\dots,e^{\lambda_n t}\big)\, V^{-1}.
$$

  → In the right coordinates the system **decouples into $n$ independent scalar systems**, each evolving as $e^{\lambda_i t}$. These are the system's **modes**.

- A complex eigenvalue $\lambda_i = \sigma_i + j\omega_i$ contributes $e^{\sigma_i t}\big(\cos\omega_i t + j\sin\omega_i t\big)$.

  → The **real part** $\sigma_i$ sets growth/decay; the **imaginary part** $\omega_i$ sets oscillation frequency.

- **Stability (continuous):** asymptotically stable $\iff$ every eigenvalue has $\mathrm{Re}(\lambda_i) < 0$ (all modes decay).

### 🧩 Discrete-Time Response and Stability

- For $x[k+1] = A_d\,x[k] + B_d\,u[k]$, induction gives (see [03 · Discrete-Time Models](03_Discrete_Time_State_Space_Models_and_Simulation.ipynb)):

$$
x[k] = A_d^{\,k}\,x[0] + \sum_{j=0}^{k-1} A_d^{\,k-1-j} B_d\, u[j].
$$

  → Same structure as continuous time, but the propagator is the **matrix power** $A_d^{\,k}$ instead of the exponential $e^{At}$, and the integral becomes a sum.

- Under zero-order-hold sampling with period $\Delta t$, $A_d = e^{A\Delta t}$, so eigenvalues map as $\lambda_i^{(d)} = e^{\lambda_i \Delta t}$.

  → The stable region maps from "left half-plane" to "**inside the unit circle**": discrete stability needs $|\lambda_i^{(d)}| < 1$.

### 🧩 Observability — "Can I reconstruct the state from the outputs?"

- $(A,C)$ is **observable** if the initial state $x(0)$ can be uniquely determined from a finite record of outputs $z(t)$ (inputs known).

- Test with the **observability matrix**

$$
\mathcal{O} = \begin{bmatrix} C \\ CA \\ CA^2 \\ \vdots \\ CA^{\,n-1}\end{bmatrix} \in \mathbb{R}^{mn \times n}, \qquad \text{observable} \iff \operatorname{rank}(\mathcal{O}) = n.
$$

  → Stack the output map $C$ together with the output map "seen through the dynamics" $CA, CA^2,\dots$. If together they touch **every direction of state space** (full rank), no hidden mode escapes the sensor.

- **→ Intuition for the KF:** an unobservable direction receives no information from measurements, so its error covariance never shrinks there — the filter stays maximally uncertain in that direction. **Observability is what lets the correction step actually reduce uncertainty.**

### 🧩 Controllability — "Can the inputs (or noise) reach every state?"

- $(A,B)$ is **controllable** if some input sequence can drive the state from any $x(0)$ to any target in finite time.

- Test with the **controllability matrix**

$$
\mathcal{C} = \begin{bmatrix} B & AB & A^2 B & \cdots & A^{\,n-1}B \end{bmatrix} \in \mathbb{R}^{n \times nm}, \qquad \text{controllable} \iff \operatorname{rank}(\mathcal{C}) = n.
$$

  → The columns show which directions the input can push **directly** ($B$) and **through the dynamics** ($AB, A^2B,\dots$). Full rank ⇒ the input's influence eventually spreads to every state direction.

---

### 🧩 Why This Matters for the Kalman Filter

- The KF's true "input" is the **process noise** $w_k$ entering through $B_w$ / $\Sigma_{\tilde w}$. Controllability of $\big(A,\ \Sigma_{\tilde w}^{1/2}\big)$ decides whether noise excites — and so keeps *uncertain* — every state direction.

- The conditions relax to **detectability** (every *unstable* mode is observable) and **stabilizability** (every *unstable* mode is controllable).

  → Exactly the conditions under which the KF error covariance converges to a constant — the **steady-state Kalman filter** (see [10 · KF Extensions](10_KF_Extensions_FaultDetection_SteadyState_Smoothing.ipynb)). No detectability ⇒ some error blows up; no stabilizability ⇒ the Riccati recursion may fail to settle.

- **→ Intuition:** observability feeds *information in* (measurements shrink covariance); controllability + process noise feed *uncertainty in* (prediction grows covariance). The KF's steady state is the **balance point** between the two.

### 🧩 Summary

- Continuous $x(t) = e^{A(t-t_0)}x(t_0) + \int e^{A(t-\tau)}Bu(\tau)\,d\tau$ and discrete $x[k] = A_d^{k}x[0] + \sum A_d^{k-1-j}B_d u[j]$ both **propagate state + accumulate input**.

- The **state-transition matrix** $\Phi = e^{A\Delta t} = A_d$ is what the KF applies to mean and covariance each prediction step.

- **Modes** = eigenvalues of $A$: real part → decay/growth, imaginary part → oscillation. Stability = left-half-plane (CT) or inside-unit-circle (DT).

- **Observability** ($\operatorname{rank}\mathcal{O}=n$): measurements can pin down the state. **Controllability** ($\operatorname{rank}\mathcal{C}=n$): inputs/noise reach the state.

- Their relaxed forms **detectability + stabilizability** guarantee a **steady-state Kalman filter** exists.

---
*Next: [06 · Stochastic Processes, White Noise & Propagating Uncertainty](06_Stochastic_Processes_and_Propagating_Uncertainty.ipynb).*